# 01 | Gaia DR3 data preprocessing
**Gaia DR3 Catalogue Spectral-Type Classification with Ensemble Learning**  
Alberto Gudiño-Ochoa · Raquel Ochoa-Ornelas

**Purpose.** Load the author-supplied Kaggle CSV, validate Gaia identifiers, apply the reported quality criteria, construct observed and extinction-corrected photometry, and prepare the FLAME-complete and class-balanced cohorts. The pipeline deliberately separates measured photometry from inferred catalogue products.

**Input:** `data/dataGaia2.csv` (configure another location in `config/experiment.json`).  
**Output:** `outputs/preprocessed.joblib` and `outputs/class_distribution.csv`. No catalogue data are included in this repository.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if not (ROOT / "config" / "experiment.json").exists():
    ROOT = ROOT.parent
if not (ROOT / "config" / "experiment.json").exists():
    raise FileNotFoundError("Start Jupyter from the repository root or notebooks/ folder.")
sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
from IPython.display import display
from src.gaia_pipeline import *
cfg = load_config(ROOT)
np.random.seed(cfg["random_state"])
print("Repository:", ROOT)

## Input validation
The `Source` field is initially loaded as a string to preserve exact 64-bit Gaia identifiers. The input is checked for missing required columns and duplicate sources before any filtering.

In [ ]:
data = load_input_csv(ROOT / cfg["data_file"])
print("Input sources:", f"{len(data):,}")
print("Unique Gaia identifiers:", f"{data['Source'].nunique():,}")
display(data["SpType-ELS"].value_counts().reindex(cfg["spectral_classes"], fill_value=0).rename("Sources"))

## Catalogue filtering and photometric transformations
Criteria: `Plx > 0`, finite `Dist > 0`, `RUWE < 1.4`, finite G/BP/RP magnitudes and band extinctions, `Pstar > 0.9`, and `PGal < 0.1`. The absolute magnitude formula is $M_X=m_X-(5\log_{10}D-5)-A_X$, with $D$ in parsecs. Three observed colours and three corrected colours are calculated directly from their respective magnitudes.

In [ ]:
clean = quality_cut_gaia(data, cfg)
ready = add_photometric_features(clean)
flame = ready.dropna(subset=FLAME_COLUMNS).reset_index(drop=True)
balanced = balanced_subsample(flame, cfg["class_cap"], cfg["random_state"])
distribution = pd.DataFrame({
    "Original": data["SpType-ELS"].value_counts(),
    "Quality filtered": clean["SpType-ELS"].value_counts(),
    "FLAME complete": flame["SpType-ELS"].value_counts(),
    "Balanced": balanced["SpType-ELS"].value_counts()
}).reindex(cfg["spectral_classes"]).fillna(0).astype(int)
display(distribution)
assert data["Source"].is_unique and flame["Source"].is_unique and balanced["Source"].is_unique
assert set(balanced["Source"]).issubset(set(flame["Source"]))
assert (distribution["Balanced"] == cfg["class_cap"]).all(), "A spectral class has fewer sources than the original cap."
print(f"Quality filtered: {len(clean):,} | FLAME complete: {len(flame):,} | Balanced: {len(balanced):,}")

## Save reproducible inputs
`preprocessed.joblib` contains the FLAME-complete cohort and the separate 35,000-source balanced cohort. Subsequent notebooks read this file; they do **not** repeat filtering with potentially different selection criteria. These outputs are excluded from Git.

In [ ]:
save_object({"flame": flame, "balanced": balanced}, ROOT, "preprocessed.joblib")
save_table(distribution.reset_index(names="Spectral class"), ROOT, "class_distribution.csv")
print("Saved preprocessing outputs to", output_dir(ROOT))